## Hybrid Search: Combining Dense and Sparse Matrices

In [13]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever ## EnsembleRetriever is used to combine Dense and Sparse Retriever
from langchain_core.documents import Document



In [7]:

docs = [
Document(page_content="LangChain helps build LLM applications."),
Document(page_content="Pinecone is a vector database for semantic search."),
Document(page_content="The Eiffel Tower is located in Paris."),
Document(page_content="LangChain can be used to develop agentic ai application."),
Document(page_content="LangChain has different types of retrievers.")

]

In [8]:
## Step2: Dense Retriever (FAISS + Hugging Face)

embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embeddings
)
dense_retriever = dense_vectorstore.as_retriever(
    search_kwargs = {"k": 3},
    serach_type = "similarity"
)

dense_vectorstore.save_local("Dense_Vectorstore")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3338.23it/s]


In [15]:
## Step 3: Creating a Sparse Retriever

sparse_retriver = BM25Retriever.from_documents(docs)
sparse_retriver.k = 3 ## top k documents to retrieve from the retriever

In [16]:
## Step 4: Combining Dense Retriver and Sparse Retriever using Ensemble Retriever

hybrid_retriver = EnsembleRetriever(
    retrievers = [dense_retriever,sparse_retriver],
    weights = [0.7,0.3]
)

In [17]:
hybrid_retriver

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f8dcd308ad0>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7f8dcd30a900>, k=3)], weights=[0.7, 0.3])

In [20]:
query = "How can i build an llm application?"
result = hybrid_retriver.invoke(query)
for i , doc in enumerate(result):
    print(f"Document: {i+1}\n {doc.page_content}\n")

Document: 1
 LangChain helps build LLM applications.

Document: 2
 LangChain can be used to develop agentic ai application.

Document: 3
 Pinecone is a vector database for semantic search.

Document: 4
 LangChain has different types of retrievers.



In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [22]:
llm = ChatOpenAI(
    model = "gpt-5-mini-2025-08-07",
    temperature = 0.4,
    max_tokens = 500
)

In [23]:
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

In [24]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [25]:
document_chain= create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=

In [26]:
rag_chain = create_retrieval_chain(hybrid_retriver,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f8dcd308ad0>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7f8dcd30a900>, k=3)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="Y

In [27]:
input = {"input": "How I can build a LLM application?"}

response = rag_chain.invoke(input)

print(f"Answer: \n {response['answer']}")

print("Sources: \n")
for i, doc in enumerate(response['context']):
   
    print(f"Doc: {i + 1}\n Content: {doc.page_content}\n")

Answer: 
 Use LangChain to orchestrate LLM calls, chains, and application logic. For agentic behavior (tools, planning, decision-making) build agents with LangChain's agent features. Store and query embeddings for semantic search with a vector DB like Pinecone.
Sources: 

Doc: 1
 Content: LangChain helps build LLM applications.

Doc: 2
 Content: LangChain can be used to develop agentic ai application.

Doc: 3
 Content: Pinecone is a vector database for semantic search.

